In [ ]:
# --- bootstrap: anchor to the repository root, wherever this notebook was opened from ---
# Notebooks live two levels deep under notebooks/, so the cwd-relative path logic below needs the
# root established first. Keyed on pytest.ini, which is not tied to any folder-naming decision.
import os
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pytest.ini").exists())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"repo root: {_root}")

# EDA: RavenPack Equity Sentiment vs S&P 500 Individual Stock Returns

**Academic research only. Not investment advice.**

This notebook operates at the individual-stock level (as opposed to market-index ETF proxies).
It aggregates RavenPack `rpa_djpr_equities` sentiment per company per trading session and joins
to CRSP daily returns for S&P 500 constituents over 2022–2023.

Key differences from `Basic_EDA_Analysis.ipynb`:
- Uses equity tables (`rpa_djpr_equities_*`) instead of global macro tables
- Sentiment is per company, not market-wide
- Returns are for individual stocks (CRSP `permno`), not ETFs
- Links RavenPack entities to CRSP via 8-char CUSIP match

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wrds

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.max_columns = 80
pd.options.display.max_rows = 60
sns.set_theme(style="whitegrid", context="notebook")

START_DATE = "2022-01-01"
END_DATE   = "2023-12-31"
YEARS      = range(2022, 2024)

RELEVANCE_MIN       = 90
EVENT_RELEVANCE_MIN = 90
MARKET_CLOSE_ET     = "16:00:00"

ET_TS = "((timestamp_utc AT TIME ZONE 'UTC') AT TIME ZONE 'America/New_York')"

print(f"Window: {START_DATE} to {END_DATE}")

## 1. Connect and inspect schemas

In [ ]:
db = wrds.Connection()

# Equity table schema
eq_preview = db.get_table(library="rpna", table="rpa_djpr_equities_2022", obs=3)
print("Equity table columns:")
print(list(eq_preview.columns))

# Company mapping table schema
map_preview = db.get_table(library="rpna", table="wrds_rpa_company_mappings", obs=3)
print("\nCompany mapping columns:")
print(list(map_preview.columns))
display(map_preview.head())

## 2. Source universe (rank-1, non-blog)

In [ ]:
def sql_string_list(values):
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)

raw_sources = db.raw_sql("""
    SELECT rp_entity_id, data_type, data_value
    FROM rpna.rpa_source_list
    WHERE data_type IN ('ENTITY_NAME', 'PUBLICATION_TYPE', 'SOURCE_RANK')
""")

sources_df = (
    raw_sources
    .pivot(index="rp_entity_id", columns="data_type", values="data_value")
    .reset_index()
    .rename(columns={"ENTITY_NAME": "source_name", "PUBLICATION_TYPE": "source_type", "SOURCE_RANK": "source_rank"})
)
sources_df.columns.name = None
sources_df["source_rank"] = pd.to_numeric(sources_df["source_rank"], errors="coerce")

valid_source_ids = (
    sources_df
    .loc[sources_df["source_rank"].eq(1) & sources_df["source_type"].ne("BLOG")]
    ["rp_entity_id"].dropna().astype(str).tolist()
)
source_id_sql = sql_string_list(valid_source_ids)
print(f"Rank-1 non-blog sources: {len(valid_source_ids)}")

## 3. S&P 500 constituents → RavenPack entity mapping

`wrds_rpa_company_mappings` has `cusip` (9-char) and `ticker` but no `permno`.
Join via `LEFT(cusip, 8)` matched to CRSP `stocknames.ncusip` (8-char, no check digit).
CUSIP matching is more stable than ticker matching because tickers get reused over time.

In [ ]:
# Step 1: S&P 500 permnos + 8-char CUSIPs from CRSP
sp500_crsp_df = db.raw_sql(f"""
    SELECT DISTINCT sp.permno::int AS permno, n.ticker, n.ncusip AS cusip8
    FROM crsp.msp500list sp
    JOIN crsp.stocknames n ON sp.permno = n.permno
    WHERE sp.start  <= DATE '{END_DATE}'
      AND COALESCE(sp.ending,    DATE '9999-12-31') >= DATE '{START_DATE}'
      AND n.namedt  <= DATE '{END_DATE}'
      AND COALESCE(n.nameenddt,  DATE '9999-12-31') >= DATE '{START_DATE}'
      AND n.ncusip IS NOT NULL
""")
sp500_crsp_df = sp500_crsp_df.drop_duplicates("permno").reset_index(drop=True)
print(f"S&P 500 CRSP permnos: {sp500_crsp_df['permno'].nunique()}")

# Step 2: RavenPack entities matched by 8-char CUSIP prefix
cusip8_list = sp500_crsp_df["cusip8"].dropna().unique().tolist()
rp_map_df = db.raw_sql(f"""
    SELECT DISTINCT rp_entity_id, LEFT(cusip, 8) AS cusip8, ticker
    FROM rpna.wrds_rpa_company_mappings
    WHERE LEFT(cusip, 8) IN ({sql_string_list(cusip8_list)})
      AND entity_type = 'COMP'
""")
print(f"RavenPack rows with CUSIP match: {len(rp_map_df)}")

# Step 3: merge → (rp_entity_id, permno, ticker)
sp500_map_df = (
    sp500_crsp_df
    .merge(rp_map_df[["rp_entity_id", "cusip8"]], on="cusip8", how="inner")
    .drop_duplicates(subset=["rp_entity_id", "permno"])
    .reset_index(drop=True)
)

entity_ids_sql = sql_string_list(sp500_map_df["rp_entity_id"].unique())
permnos_sql    = sql_string_list(sp500_map_df["permno"].unique())

print(f"Mapped: {sp500_map_df['rp_entity_id'].nunique()} RavenPack entities → {sp500_map_df['permno'].nunique()} CRSP permnos")
display(sp500_map_df.head())

## 4. Aggregate equity sentiment per company per trading session

Same after-hours logic as the macro notebook: articles published after 4 PM ET are assigned to the next calendar day.

In [ ]:
def fetch_equity_sentiment_for_year(year):
    table = f"rpna.rpa_djpr_equities_{year}"
    query = f"""
        SELECT
            rp_entity_id,
            CASE
                WHEN {ET_TS}::time < TIME '{MARKET_CLOSE_ET}'
                    THEN {ET_TS}::date
                ELSE ({ET_TS}::date + INTERVAL '1 day')::date
            END AS signal_calendar_date,
            COUNT(*)::bigint                                                          AS event_count,
            COUNT(DISTINCT rp_story_id)::bigint                                      AS story_count,
            AVG(event_sentiment_score)::float                                        AS mean_ess,
            SUM(CASE WHEN event_sentiment_score >  0.05 THEN 1 ELSE 0 END)::bigint  AS positive_count,
            SUM(CASE WHEN event_sentiment_score < -0.05 THEN 1 ELSE 0 END)::bigint  AS negative_count,
            SUM(CASE WHEN event_sentiment_score BETWEEN -0.05 AND 0.05
                     THEN 1 ELSE 0 END)::bigint                                      AS neutral_count
        FROM {table}
        WHERE rpa_date_utc BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'
          AND relevance        >= {RELEVANCE_MIN}
          AND event_relevance  >= {EVENT_RELEVANCE_MIN}
          AND rp_source_id IN ({source_id_sql})
          AND rp_entity_id IN ({entity_ids_sql})
          AND timestamp_utc IS NOT NULL
          AND event_sentiment_score IS NOT NULL
        GROUP BY rp_entity_id, 2
        ORDER BY rp_entity_id, 2
    """
    return db.raw_sql(query)

equity_sent_parts = []
for year in YEARS:
    print(f"Pulling equity sentiment for {year}...")
    equity_sent_parts.append(fetch_equity_sentiment_for_year(year))

equity_sent_raw = pd.concat(equity_sent_parts, ignore_index=True)
equity_sent_raw["signal_calendar_date"] = pd.to_datetime(equity_sent_raw["signal_calendar_date"])

print(f"\nRaw entity-day rows: {len(equity_sent_raw):,}")
print(f"Unique entities:     {equity_sent_raw['rp_entity_id'].nunique()}")
display(equity_sent_raw.head())

## 5. CRSP daily returns for S&P 500 constituents

In [ ]:
crsp_query = f"""
    SELECT
        d.dlycaldt    AS session_date,
        d.permno      AS permno,
        d.dlyret      AS daily_return,
        ABS(d.dlyprc) AS price,
        d.dlyvol      AS volume
    FROM crsp.dsf_v2 d
    WHERE d.permno IN ({permnos_sql})
      AND d.dlycaldt BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'
    ORDER BY d.permno, d.dlycaldt
"""

try:
    crsp_df = db.raw_sql(crsp_query)
    crsp_source = "crsp.dsf_v2"
except Exception:
    legacy = crsp_query.replace("crsp.dsf_v2", "crsp.dsf").replace("dlycaldt", "date") \
                       .replace("dlyret", "ret").replace("dlyprc", "prc").replace("dlyvol", "vol")
    crsp_df = db.raw_sql(legacy)
    crsp_source = "crsp.dsf (legacy)"

crsp_df["session_date"] = pd.to_datetime(crsp_df["session_date"])
crsp_df["daily_return"] = pd.to_numeric(crsp_df["daily_return"], errors="coerce")
crsp_df["price"]        = pd.to_numeric(crsp_df["price"],        errors="coerce")
crsp_df["volume"]       = pd.to_numeric(crsp_df["volume"],       errors="coerce")
crsp_df = crsp_df.drop_duplicates(["permno", "session_date"])

# Forward returns: 1-day and 5-day
crsp_df = crsp_df.sort_values(["permno", "session_date"]).reset_index(drop=True)
for horizon in [1, 5]:
    shifted = pd.Series(1.0, index=crsp_df.index)
    for lag in range(1, horizon + 1):
        shifted *= 1 + crsp_df.groupby("permno")["daily_return"].shift(-lag)
    crsp_df[f"fwd_{horizon}d_return"]   = shifted - 1
    crsp_df[f"fwd_{horizon}d_positive"] = (crsp_df[f"fwd_{horizon}d_return"] > 0).astype(float)
    crsp_df.loc[crsp_df[f"fwd_{horizon}d_return"].isna(), f"fwd_{horizon}d_positive"] = np.nan

trading_sessions = crsp_df["session_date"].drop_duplicates().sort_values()

print(f"CRSP source: {crsp_source}")
print(f"Rows: {len(crsp_df):,}  |  Permnos: {crsp_df['permno'].nunique()}  |  Sessions: {crsp_df['session_date'].nunique()}")
display(crsp_df.head())

## 6. Align sentiment to trading sessions and join to returns

In [ ]:
def map_to_next_trading_session(dates, sessions):
    session_arr = sessions.sort_values().to_numpy(dtype="datetime64[ns]")
    target_arr  = pd.to_datetime(dates).to_numpy(dtype="datetime64[ns]")
    positions   = np.searchsorted(session_arr, target_arr, side="left")
    mapped      = np.full(len(target_arr), np.datetime64("NaT"), dtype="datetime64[ns]")
    valid       = positions < len(session_arr)
    mapped[valid] = session_arr[positions[valid]]
    return pd.to_datetime(mapped)

equity_sent = equity_sent_raw.copy()
equity_sent["session_date"] = map_to_next_trading_session(
    equity_sent["signal_calendar_date"], trading_sessions
)
equity_sent = equity_sent.dropna(subset=["session_date"])

# Attach permno + ticker
entity_to_permno = sp500_map_df[["rp_entity_id", "permno", "ticker"]].drop_duplicates("rp_entity_id")
equity_sent = equity_sent.merge(entity_to_permno, on="rp_entity_id", how="inner")

# Sentiment bucket
equity_sent["sentiment_bucket"] = np.select(
    [equity_sent["mean_ess"] > 0.05, equity_sent["mean_ess"] < -0.05],
    ["positive", "negative"],
    default="neutral",
)

# Join to CRSP returns
panel_df = equity_sent.merge(
    crsp_df[["permno", "session_date", "daily_return", "price", "volume",
              "fwd_1d_return", "fwd_1d_positive", "fwd_5d_return", "fwd_5d_positive"]],
    on=["permno", "session_date"],
    how="inner",
)
panel_df["year"] = panel_df["session_date"].dt.year

print(f"Panel shape: {panel_df.shape}")
print(f"Unique stocks: {panel_df['permno'].nunique()}  |  Sessions: {panel_df['session_date'].nunique()}")
display(panel_df.head())

## 7. Validation

In [ ]:
assert panel_df.duplicated(["permno", "session_date"]).sum() == 0, "Duplicate permno-session rows"
assert (equity_sent_raw["story_count"] <= equity_sent_raw["event_count"]).all(), "story_count > event_count"

# Forward return spot check: fwd_1d_return must equal next day's daily_return
check = crsp_df.sort_values(["permno", "session_date"]).copy()
check["next_return"] = check.groupby("permno")["daily_return"].shift(-1)
valid = check[["fwd_1d_return", "next_return"]].dropna()
assert np.allclose(valid["fwd_1d_return"], valid["next_return"], atol=1e-10), "fwd_1d_return mismatch"

print("Validation passed.")
print(panel_df[["mean_ess", "fwd_1d_return", "fwd_5d_return", "daily_return"]].describe().round(4))

## 8. EDA plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Sentiment score distribution
sns.histplot(panel_df["mean_ess"].dropna(), bins=60, kde=True, ax=axes[0], color="#54A24B")
axes[0].axvline( 0.05, color="green", linestyle="--", linewidth=1, label="+0.05")
axes[0].axvline(-0.05, color="red",   linestyle="--", linewidth=1, label="-0.05")
axes[0].set_title("Equity sentiment score distribution")
axes[0].set_xlabel("Mean event sentiment score")
axes[0].legend()

# Sentiment bucket counts
bucket_counts = panel_df["sentiment_bucket"].value_counts().reset_index()
bucket_counts.columns = ["bucket", "count"]
sns.barplot(data=bucket_counts, x="bucket", y="count", ax=axes[1],
            order=["positive", "neutral", "negative"],
            palette={"positive": "#54A24B", "neutral": "#4C78A8", "negative": "#E45756"})
axes[1].set_title("Company-day observations by sentiment bucket")
axes[1].set_xlabel("")

# Average 1-day forward return by bucket
bucket_ret = (
    panel_df.dropna(subset=["fwd_1d_return"])
    .groupby("sentiment_bucket")["fwd_1d_return"]
    .mean().reset_index()
    .rename(columns={"fwd_1d_return": "avg_fwd_1d_return"})
)
sns.barplot(data=bucket_ret, x="sentiment_bucket", y="avg_fwd_1d_return", ax=axes[2],
            order=["positive", "neutral", "negative"],
            palette={"positive": "#54A24B", "neutral": "#4C78A8", "negative": "#E45756"})
axes[2].set_title("Average next-day return by sentiment bucket")
axes[2].set_xlabel("")
axes[2].axhline(0, color="black", linewidth=0.8, linestyle="--")

plt.tight_layout()
plt.show()

# Summary table
bucket_summary = (
    panel_df.dropna(subset=["fwd_1d_return", "fwd_5d_return"])
    .groupby("sentiment_bucket", as_index=False)
    .agg(
        observations=("session_date", "count"),
        avg_fwd_1d_return=("fwd_1d_return", "mean"),
        avg_fwd_5d_return=("fwd_5d_return", "mean"),
        pct_up_next_day=("fwd_1d_positive", "mean"),
    )
)
display(bucket_summary)

In [ ]:
# Correlation heatmap: sentiment features vs returns
corr_cols = [
    "event_count", "story_count",
    "mean_ess", "positive_count", "negative_count",
    "daily_return", "fwd_1d_return", "fwd_5d_return", "volume",
]
corr_matrix = (
    panel_df[corr_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(how="all")
    .corr(numeric_only=True)
)

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="vlag", center=0, linewidths=0.5)
plt.title("Correlation: equity sentiment features vs individual stock returns (2022-2023)")
plt.tight_layout()
plt.show()